# Chapter 18 Companion Notebook: Classic Generative Models in Business Analytics

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*  
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song  
**Notebook author:** Hyunhwan Aiden Lee  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/notebooks/Ch18_Classic_Generative_Models.ipynb)

This notebook accompanies Chapter 18 of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).




Classroom note: this notebook uses synthetic customer data and small PyTorch models so students can run the workflow in Colab without a paid API or external dataset.

Copyright 2026 to present.

## How to use this notebook

Run the cells from top to bottom. Treat markdown sections as short lecture notes and code sections as live demos. If a section runs slowly, keep `FAST_MODE = True`, reduce `N_CUSTOMERS`, or reduce the VAE, cVAE, and GAN epoch settings in the setup cell.

## Why this matters (business framing)

Predictive analytics asks what outcome is likely. Generative modeling asks what data could plausibly occur if a model has learned the structure of the observed distribution. That shift matters for business analytics because many decisions require plausible variation rather than one score: synthetic customer profiles for simulation, visual alternatives for concept testing, augmentation under data scarcity, and stress tests for rare but important cases.

This notebook follows the practical logic of Chapter 18. We will train a basic variational autoencoder, add conditioning for controllable generation, train a small GAN in a two-dimensional representation space, diagnose mode collapse, evaluate generation with a scorecard, test downstream utility with a TSTR protocol, and finish with privacy, leakage, and governance checks. The emphasis is not on producing impressive samples. The emphasis is on deciding whether synthetic samples are useful, diverse, aligned, and safe enough for a business workflow.

## Agenda

1. Setup and reproducibility
2. Synthetic customer profile data for generation
3. Predictive versus generative framing with a simple baseline generator
4. Variational autoencoder: reconstruction plus latent regularization
5. Conditional VAE: controllable generation by segment
6. GAN: generator versus discriminator in a small representation space
7. Mode collapse as a coverage failure
8. Image-to-image translation as a problem setup
9. Evaluation scorecard: fidelity, coverage, alignment, and constraints
10. Utility validation with TSTR: train on synthetic, test on real
11. Privacy and memorization diagnostics
12. Bridge to diffusion models: one-step generation versus iterative denoising
13. Governance artifacts and exercises

## Learning objectives (measurable)

By the end of this notebook, you should be able to explain the difference between predictive and generative modeling, train a small VAE and interpret reconstruction and KL terms, use a conditional VAE to generate segment-specific customer profiles, train a small GAN and inspect adversarial training dynamics, detect mode collapse as missing coverage, evaluate synthetic data with more than one metric, run a TSTR utility check, compute a nearest-neighbor memorization diagnostic, and write a governance-oriented model card for a generative workflow.

## Connection map

Chapter 17 introduced autoencoders as representation learners. Chapter 18 turns that representation idea into generation. A VAE makes the encoder stochastic so that the latent space becomes sampleable. A GAN replaces reconstruction with an adversarial signal from a discriminator. Both families can support business workflows, but only when generation is validated against purpose, coverage, constraints, and governance requirements.

In [ ]:
# ============================================================
# 1. Setup and reproducibility
# - install missing packages if needed
# - import libraries
# - set seeds
# - configure output folders
# ============================================================
import os
import sys
import math
import json
import time
import random
import warnings
import importlib
import subprocess
from pathlib import Path


def ensure(pkg_import_name, pip_name=None):
    """Install a package only if it is missing."""
    try:
        importlib.import_module(pkg_import_name)
    except Exception:
        pip_target = pip_name or pkg_import_name
        print(f"Installing {pip_target} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_target])


for import_name, pip_name in [
    ("numpy", None),
    ("pandas", None),
    ("sklearn", "scikit-learn"),
    ("matplotlib", None),
    ("torch", None),
]:
    ensure(import_name, pip_name)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

SEED = 18
FAST_MODE = True
N_CUSTOMERS = 3200 if FAST_MODE else 6000
BATCH_SIZE = 128
LATENT_DIM = 4
EPOCHS_VAE = 35 if FAST_MODE else 80
EPOCHS_CVAE = 40 if FAST_MODE else 90
EPOCHS_GAN = 450 if FAST_MODE else 900
GAN_BATCH_SIZE = 256

OUTPUT_DIR = Path("ch18_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
try:
    torch.set_num_threads(1)
except Exception:
    pass

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"FAST_MODE: {FAST_MODE}")
print(f"Synthetic customers: {N_CUSTOMERS:,}")

## Utility functions

These helpers keep the main sections focused on modeling decisions. The most important idea is that a generator should be evaluated as a distributional tool, not by one attractive sample.

In [ ]:
# ============================================================
# Utility functions for display, plotting, metrics, and tensors
# ============================================================

def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def to_tensor(x, dtype=torch.float32):
    return torch.tensor(x, dtype=dtype, device=device)


def one_hot_np(y, num_classes):
    y = np.asarray(y, dtype=int)
    out = np.zeros((len(y), num_classes), dtype=np.float32)
    out[np.arange(len(y)), y] = 1.0
    return out


def plot_training_history(history, title, keys=None):
    keys = keys or list(history.keys())
    plt.figure(figsize=(8, 4.5))
    for key in keys:
        if key in history:
            plt.plot(history[key], label=key)
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()


def plot_2d(real_points, synthetic_points, title, real_label="Real", synthetic_label="Synthetic", max_points=900):
    rng = np.random.default_rng(SEED)
    r_idx = rng.choice(len(real_points), size=min(max_points, len(real_points)), replace=False)
    s_idx = rng.choice(len(synthetic_points), size=min(max_points, len(synthetic_points)), replace=False)
    plt.figure(figsize=(6.5, 5.5))
    plt.scatter(real_points[r_idx, 0], real_points[r_idx, 1], s=12, alpha=0.35, label=real_label)
    plt.scatter(synthetic_points[s_idx, 0], synthetic_points[s_idx, 1], s=12, alpha=0.35, label=synthetic_label)
    plt.title(title)
    plt.xlabel("Dimension 1")
    plt.ylabel("Dimension 2")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()


def safe_corr(x):
    corr = np.corrcoef(x, rowvar=False)
    corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
    return corr


def entropy_from_shares(shares):
    shares = np.asarray(shares, dtype=float)
    shares = shares[shares > 0]
    if len(shares) == 0:
        return 0.0
    return float(-(shares * np.log(shares)).sum() / np.log(max(2, len(shares))))

## 2. Synthetic customer profile data

We use synthetic customer profiles because this notebook is meant to be safe, shareable, and runnable in class. The setting is a subscription retail platform that wants to generate plausible customer profiles for internal simulation. The data include segment labels, transformed monetary value, engagement rates, service friction, and a delayed churn label. The delayed label is included for diagnostics, but the generative models below focus on learning the distribution of customer profiles.

In [ ]:
# ============================================================
# 2. Synthetic customer profile data for generation
# ============================================================
SEGMENTS = [
    "Budget Value Seekers",
    "Loyal Premium",
    "At Risk Service Heavy",
    "New Explorers",
]
SEGMENT_PROBS = np.array([0.34, 0.26, 0.24, 0.16])
SEGMENT_PROBS = SEGMENT_PROBS / SEGMENT_PROBS.sum()

CHANNELS = ["Email", "Mobile App", "Search", "Store"]


def beta_sample(rng, a, b, n):
    return np.clip(rng.beta(a, b, n), 0.0, 1.0)


def simulate_customers(n=N_CUSTOMERS, seed=SEED):
    rng = np.random.default_rng(seed)
    segment = rng.choice(SEGMENTS, size=n, p=SEGMENT_PROBS)
    month = rng.integers(1, 13, size=n)
    as_of_date = pd.to_datetime([f"2024-{m:02d}-01" for m in month])
    channel = rng.choice(CHANNELS, size=n, p=[0.32, 0.34, 0.21, 0.13])

    recency_days = np.zeros(n)
    sessions_30d = np.zeros(n)
    orders_90d = np.zeros(n)
    log_monetary_90d = np.zeros(n)
    support_tickets_90d = np.zeros(n)
    discount_share = np.zeros(n)
    email_open_rate = np.zeros(n)
    mobile_share = np.zeros(n)
    return_rate = np.zeros(n)
    tenure_months = np.zeros(n)

    for s in SEGMENTS:
        idx = np.where(segment == s)[0]
        m = len(idx)
        if s == "Budget Value Seekers":
            recency_days[idx] = np.clip(rng.gamma(2.3, 9.5, m), 0, 120)
            sessions_30d[idx] = rng.poisson(8.0, m)
            orders_90d[idx] = rng.poisson(3.4, m)
            log_monetary_90d[idx] = np.clip(rng.normal(4.55, 0.45, m), 0, None)
            support_tickets_90d[idx] = rng.poisson(0.45, m)
            discount_share[idx] = beta_sample(rng, 6.0, 3.0, m)
            email_open_rate[idx] = beta_sample(rng, 3.0, 6.5, m)
            mobile_share[idx] = beta_sample(rng, 6.5, 3.0, m)
            return_rate[idx] = beta_sample(rng, 1.6, 20.0, m)
            tenure_months[idx] = np.clip(rng.gamma(2.5, 7.0, m), 0, 72)
        elif s == "Loyal Premium":
            recency_days[idx] = np.clip(rng.gamma(1.6, 5.0, m), 0, 120)
            sessions_30d[idx] = rng.poisson(17.0, m)
            orders_90d[idx] = rng.poisson(7.5, m)
            log_monetary_90d[idx] = np.clip(rng.normal(5.95, 0.42, m), 0, None)
            support_tickets_90d[idx] = rng.poisson(0.25, m)
            discount_share[idx] = beta_sample(rng, 2.0, 8.0, m)
            email_open_rate[idx] = beta_sample(rng, 6.0, 4.0, m)
            mobile_share[idx] = beta_sample(rng, 5.0, 4.5, m)
            return_rate[idx] = beta_sample(rng, 1.3, 28.0, m)
            tenure_months[idx] = np.clip(rng.gamma(4.0, 8.0, m), 0, 96)
        elif s == "At Risk Service Heavy":
            recency_days[idx] = np.clip(rng.gamma(3.8, 13.0, m), 0, 150)
            sessions_30d[idx] = rng.poisson(4.5, m)
            orders_90d[idx] = rng.poisson(1.4, m)
            log_monetary_90d[idx] = np.clip(rng.normal(3.65, 0.55, m), 0, None)
            support_tickets_90d[idx] = rng.poisson(1.8, m)
            discount_share[idx] = beta_sample(rng, 4.0, 5.5, m)
            email_open_rate[idx] = beta_sample(rng, 1.8, 9.5, m)
            mobile_share[idx] = beta_sample(rng, 3.0, 5.0, m)
            return_rate[idx] = beta_sample(rng, 2.3, 14.0, m)
            tenure_months[idx] = np.clip(rng.gamma(3.0, 6.0, m), 0, 84)
        else:  # New Explorers
            recency_days[idx] = np.clip(rng.gamma(2.0, 7.0, m), 0, 100)
            sessions_30d[idx] = rng.poisson(6.0, m)
            orders_90d[idx] = rng.poisson(1.6, m)
            log_monetary_90d[idx] = np.clip(rng.normal(3.95, 0.50, m), 0, None)
            support_tickets_90d[idx] = rng.poisson(0.55, m)
            discount_share[idx] = beta_sample(rng, 3.5, 5.5, m)
            email_open_rate[idx] = beta_sample(rng, 2.8, 7.5, m)
            mobile_share[idx] = beta_sample(rng, 7.5, 2.5, m)
            return_rate[idx] = beta_sample(rng, 1.6, 18.0, m)
            tenure_months[idx] = np.clip(rng.gamma(1.7, 2.5, m), 0, 24)

    # A delayed churn label. It is not used as a target for generation, but it helps students see downstream utility.
    segment_churn_shift = np.select(
        [segment == "At Risk Service Heavy", segment == "Loyal Premium", segment == "New Explorers"],
        [0.95, -0.70, 0.10],
        default=0.0,
    )
    churn_logit = (
        -2.35
        + 0.030 * recency_days
        + 0.18 * support_tickets_90d
        + 3.00 * return_rate
        - 0.085 * orders_90d
        - 1.15 * email_open_rate
        - 0.015 * tenure_months
        + segment_churn_shift
    )
    churn_probability = sigmoid(churn_logit)
    churn_next_month = rng.binomial(1, churn_probability)

    df = pd.DataFrame(
        {
            "customer_id": [f"C{i:05d}" for i in range(n)],
            "as_of_date": as_of_date,
            "segment": segment,
            "primary_channel": channel,
            "recency_days": recency_days,
            "sessions_30d": sessions_30d,
            "orders_90d": orders_90d,
            "log_monetary_90d": log_monetary_90d,
            "support_tickets_90d": support_tickets_90d,
            "discount_share": discount_share,
            "email_open_rate": email_open_rate,
            "mobile_share": mobile_share,
            "return_rate": return_rate,
            "tenure_months": tenure_months,
            "churn_probability": churn_probability,
            "churn_next_month": churn_next_month,
        }
    )
    return df.sort_values(["as_of_date", "customer_id"]).reset_index(drop=True)


df = simulate_customers()
print(df.shape)
display(df.head())

In [ ]:
# A quick profile by segment. These are the patterns the generator should learn.
profile_cols = [
    "recency_days",
    "sessions_30d",
    "orders_90d",
    "log_monetary_90d",
    "support_tickets_90d",
    "discount_share",
    "email_open_rate",
    "return_rate",
    "tenure_months",
    "churn_next_month",
]
segment_profile = df.groupby("segment")[profile_cols].mean().round(3)
display(segment_profile)

## 3. Time-respecting preprocessing and a baseline generator

Even when the goal is generation, the validation split should respect the decision context. Here we train on earlier months and validate on later months. We also create a simple independent marginal generator. It samples each feature separately from the training distribution. This baseline can look good on one-variable histograms, but it breaks relationships among variables, which is a useful warning about evaluating generators too narrowly.

In [ ]:
# ============================================================
# 3. Time-respecting split and preprocessing
# ============================================================
feature_cols = [
    "recency_days",
    "sessions_30d",
    "orders_90d",
    "log_monetary_90d",
    "support_tickets_90d",
    "discount_share",
    "email_open_rate",
    "mobile_share",
    "return_rate",
    "tenure_months",
]

train_mask = df["as_of_date"].dt.month <= 8
val_mask = df["as_of_date"].dt.month.isin([9, 10])
test_mask = df["as_of_date"].dt.month >= 11

df_train = df.loc[train_mask].copy()
df_val = df.loc[val_mask].copy()
df_test = df.loc[test_mask].copy()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(df_train[feature_cols]).astype(np.float32)
X_val_scaled = scaler.transform(df_val[feature_cols]).astype(np.float32)
X_test_scaled = scaler.transform(df_test[feature_cols]).astype(np.float32)

segment_to_id = {s: i for i, s in enumerate(SEGMENTS)}
id_to_segment = {i: s for s, i in segment_to_id.items()}
y_train = df_train["segment"].map(segment_to_id).values.astype(int)
y_val = df_val["segment"].map(segment_to_id).values.astype(int)
y_test = df_test["segment"].map(segment_to_id).values.astype(int)
NUM_SEGMENTS = len(SEGMENTS)

print("Train/validation/test sizes:", len(df_train), len(df_val), len(df_test))
print("Feature dimension:", X_train_scaled.shape[1])
print("Segments:", segment_to_id)

In [ ]:
# ============================================================
# Baseline generator: sample each feature independently from training marginals
# ============================================================

def sample_independent_marginals(X_reference, n, seed=SEED):
    rng = np.random.default_rng(seed)
    columns = []
    for j in range(X_reference.shape[1]):
        columns.append(rng.choice(X_reference[:, j], size=n, replace=True))
    return np.column_stack(columns).astype(np.float32)


def raw_from_scaled(X_scaled):
    raw = pd.DataFrame(scaler.inverse_transform(X_scaled), columns=feature_cols)
    raw["monetary_90d"] = np.expm1(np.clip(raw["log_monetary_90d"], 0, None))
    return raw


independent_scaled = sample_independent_marginals(X_train_scaled, len(X_test_scaled), seed=SEED + 1)

real_corr = safe_corr(X_test_scaled)
ind_corr = safe_corr(independent_scaled)
print("Mean absolute correlation gap for independent marginals:", round(np.mean(np.abs(real_corr - ind_corr)), 3))

summary = pd.DataFrame(
    {
        "real_test_mean": X_test_scaled.mean(axis=0),
        "independent_mean": independent_scaled.mean(axis=0),
        "real_test_std": X_test_scaled.std(axis=0),
        "independent_std": independent_scaled.std(axis=0),
    },
    index=feature_cols,
).round(3)
display(summary)

In [ ]:
# Visualize how the independent baseline can preserve marginals but lose relationships.
pca_baseline = PCA(n_components=2, random_state=SEED).fit(X_train_scaled)
real_test_2d = pca_baseline.transform(X_test_scaled)
ind_2d = pca_baseline.transform(independent_scaled)
plot_2d(real_test_2d, ind_2d, "Real test profiles versus independent marginal generator")

## 4. Variational autoencoder: reconstruction plus regularization

A VAE is an autoencoder with a stochastic encoder. The encoder outputs a mean and log-variance for the latent representation. The model samples a latent vector through the reparameterization trick, then decodes it back into a profile. Training balances two forces: reconstruction, which asks the generated profile to match the input, and KL regularization, which keeps the latent space close enough to a simple prior that we can sample from it later.

In [ ]:
# ============================================================
# 4. VAE model and training loop
# ============================================================
class TabularVAE(nn.Module):
    def __init__(self, input_dim, latent_dim=LATENT_DIM, hidden_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.mu = nn.Linear(hidden_dim, latent_dim)
        self.logvar = nn.Linear(hidden_dim, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.mu(h), self.logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar


def vae_objective(x_hat, x, mu, logvar, beta):
    recon = F.mse_loss(x_hat, x, reduction="mean")
    kl = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
    return recon + beta * kl, recon.detach(), kl.detach()


def train_vae(model, X_train, X_val, epochs=EPOCHS_VAE, batch_size=BATCH_SIZE, beta_max=0.15, lr=1e-3):
    model.to(device)
    train_loader = DataLoader(TensorDataset(to_tensor(X_train)), batch_size=batch_size, shuffle=True)
    X_val_t = to_tensor(X_val)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_total": [], "train_recon": [], "train_kl": [], "val_total": []}

    for epoch in range(1, epochs + 1):
        model.train()
        beta = beta_max * min(1.0, epoch / max(1, epochs // 2))
        total_losses, recon_losses, kl_losses = [], [], []
        for (xb,) in train_loader:
            opt.zero_grad()
            x_hat, mu, logvar = model(xb)
            loss, recon, kl = vae_objective(x_hat, xb, mu, logvar, beta)
            loss.backward()
            opt.step()
            total_losses.append(loss.item())
            recon_losses.append(recon.item())
            kl_losses.append(kl.item())

        model.eval()
        with torch.no_grad():
            val_hat, val_mu, val_logvar = model(X_val_t)
            val_loss, _, _ = vae_objective(val_hat, X_val_t, val_mu, val_logvar, beta)

        history["train_total"].append(float(np.mean(total_losses)))
        history["train_recon"].append(float(np.mean(recon_losses)))
        history["train_kl"].append(float(np.mean(kl_losses)))
        history["val_total"].append(float(val_loss.item()))

    return history


set_seed(SEED)
vae = TabularVAE(input_dim=X_train_scaled.shape[1], latent_dim=LATENT_DIM, hidden_dim=64)
vae_history = train_vae(vae, X_train_scaled, X_val_scaled)
plot_training_history(vae_history, "VAE training history", keys=["train_total", "val_total", "train_recon", "train_kl"])

In [ ]:
# Sample new profiles from the VAE prior.
def sample_vae(model, n, seed=SEED):
    set_seed(seed)
    model.eval()
    with torch.no_grad():
        z = torch.randn(n, LATENT_DIM, device=device)
        x_synth = model.decode(z).cpu().numpy().astype(np.float32)
    return x_synth


vae_scaled = sample_vae(vae, len(X_test_scaled), seed=SEED + 2)
vae_raw = raw_from_scaled(vae_scaled)
print("VAE synthetic sample:")
display(vae_raw.head().round(3))

In [ ]:
# Inspect the learned latent representation. This is diagnostic, not a final evaluation.
vae.eval()
with torch.no_grad():
    mu_test, _ = vae.encode(to_tensor(X_test_scaled))
    latent_test = mu_test.cpu().numpy()

latent_2d = PCA(n_components=2, random_state=SEED).fit_transform(latent_test)
plt.figure(figsize=(7, 5.5))
for sid, s in id_to_segment.items():
    idx = y_test == sid
    plt.scatter(latent_2d[idx, 0], latent_2d[idx, 1], s=14, alpha=0.45, label=s)
plt.title("VAE latent means by observed segment")
plt.xlabel("Latent PCA dimension 1")
plt.ylabel("Latent PCA dimension 2")
plt.legend(fontsize=8)
plt.grid(alpha=0.2)
plt.show()

## 5. Conditional VAE: controllable generation by segment

Uncontrolled generation is rarely enough in business analytics. A conditional VAE adds a condition, such as a segment label, to both the encoder and the decoder. This lets the model generate profiles that are plausible under a requested condition. The managerial question is no longer just whether samples look real. It is whether they respect the requested segment or scenario.

In [ ]:
# ============================================================
# 5. Conditional VAE
# ============================================================
class ConditionalVAE(nn.Module):
    def __init__(self, input_dim, condition_dim, latent_dim=LATENT_DIM, hidden_dim=72):
        super().__init__()
        self.condition_dim = condition_dim
        self.encoder = nn.Sequential(
            nn.Linear(input_dim + condition_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.mu = nn.Linear(hidden_dim, latent_dim)
        self.logvar = nn.Linear(hidden_dim, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim + condition_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )

    def encode(self, x, c):
        h = self.encoder(torch.cat([x, c], dim=1))
        return self.mu(h), self.logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, c):
        return self.decoder(torch.cat([z, c], dim=1))

    def forward(self, x, c):
        mu, logvar = self.encode(x, c)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z, c)
        return x_hat, mu, logvar


def train_cvae(model, X_train, y_train, X_val, y_val, epochs=EPOCHS_CVAE, batch_size=BATCH_SIZE, beta_max=0.12, lr=1e-3):
    model.to(device)
    C_train = one_hot_np(y_train, NUM_SEGMENTS)
    C_val = one_hot_np(y_val, NUM_SEGMENTS)
    train_loader = DataLoader(
        TensorDataset(to_tensor(X_train), to_tensor(C_train)),
        batch_size=batch_size,
        shuffle=True,
    )
    X_val_t = to_tensor(X_val)
    C_val_t = to_tensor(C_val)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_total": [], "train_recon": [], "train_kl": [], "val_total": []}

    for epoch in range(1, epochs + 1):
        model.train()
        beta = beta_max * min(1.0, epoch / max(1, epochs // 2))
        total_losses, recon_losses, kl_losses = [], [], []
        for xb, cb in train_loader:
            opt.zero_grad()
            x_hat, mu, logvar = model(xb, cb)
            loss, recon, kl = vae_objective(x_hat, xb, mu, logvar, beta)
            loss.backward()
            opt.step()
            total_losses.append(loss.item())
            recon_losses.append(recon.item())
            kl_losses.append(kl.item())

        model.eval()
        with torch.no_grad():
            val_hat, val_mu, val_logvar = model(X_val_t, C_val_t)
            val_loss, _, _ = vae_objective(val_hat, X_val_t, val_mu, val_logvar, beta)

        history["train_total"].append(float(np.mean(total_losses)))
        history["train_recon"].append(float(np.mean(recon_losses)))
        history["train_kl"].append(float(np.mean(kl_losses)))
        history["val_total"].append(float(val_loss.item()))

    return history


set_seed(SEED)
cvae = ConditionalVAE(input_dim=X_train_scaled.shape[1], condition_dim=NUM_SEGMENTS, latent_dim=LATENT_DIM)
cvae_history = train_cvae(cvae, X_train_scaled, y_train, X_val_scaled, y_val)
plot_training_history(cvae_history, "Conditional VAE training history", keys=["train_total", "val_total", "train_recon", "train_kl"])

In [ ]:
# Generate profiles for requested segments.
def sample_cvae(model, labels, seed=SEED):
    set_seed(seed)
    labels = np.asarray(labels, dtype=int)
    C = one_hot_np(labels, NUM_SEGMENTS)
    model.eval()
    with torch.no_grad():
        z = torch.randn(len(labels), LATENT_DIM, device=device)
        x_synth = model.decode(z, to_tensor(C)).cpu().numpy().astype(np.float32)
    return x_synth


n_per_segment = 350 if FAST_MODE else 700
requested_labels = np.repeat(np.arange(NUM_SEGMENTS), n_per_segment)
cvae_balanced_scaled = sample_cvae(cvae, requested_labels, seed=SEED + 3)
cvae_balanced_raw = raw_from_scaled(cvae_balanced_scaled)
cvae_balanced_raw["requested_segment"] = [id_to_segment[i] for i in requested_labels]

selected_cols = ["recency_days", "orders_90d", "log_monetary_90d", "support_tickets_90d", "discount_share", "email_open_rate", "tenure_months"]
real_by_segment = df_train.groupby("segment")[selected_cols].mean().round(2)
synth_by_segment = cvae_balanced_raw.groupby("requested_segment")[selected_cols].mean().round(2)
print("Real training means by segment:")
display(real_by_segment)
print("cVAE generated means by requested segment:")
display(synth_by_segment.loc[SEGMENTS])

In [ ]:
# Project cVAE samples into the same PCA space used earlier.
cvae_2d = pca_baseline.transform(cvae_balanced_scaled)
plot_2d(real_test_2d, cvae_2d, "Real test profiles versus conditional VAE samples")

## 6. GAN: generator versus discriminator

A GAN trains two networks in competition. The generator transforms noise into synthetic samples. The discriminator tries to distinguish real samples from synthetic samples. In this teaching example, we train a small GAN on a two-dimensional PCA representation of customer profiles. This keeps training fast while preserving the main idea: the generator learns from an adversarial quality-control signal rather than from reconstruction.

In [ ]:
# ============================================================
# 6. Small GAN in a two-dimensional representation space
# ============================================================
real2_train = pca_baseline.transform(X_train_scaled)
real2_scaler = StandardScaler().fit(real2_train)
real2_train_z = real2_scaler.transform(real2_train).astype(np.float32)
real2_test_z = real2_scaler.transform(real_test_2d).astype(np.float32)

class Generator2D(nn.Module):
    def __init__(self, z_dim=6, hidden_dim=48):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, z):
        return self.net(z)


class Discriminator2D(nn.Module):
    def __init__(self, hidden_dim=48):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x).view(-1)


def train_gan_2d(real_points, epochs=EPOCHS_GAN, batch_size=GAN_BATCH_SIZE, z_dim=6, lr=1e-3):
    set_seed(SEED)
    G = Generator2D(z_dim=z_dim).to(device)
    D = Discriminator2D().to(device)
    opt_g = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    bce = nn.BCEWithLogitsLoss()
    real_t = to_tensor(real_points)
    history = {"d_loss": [], "g_loss": [], "d_real_score": [], "d_fake_score": []}

    for epoch in range(1, epochs + 1):
        idx = torch.randint(0, real_t.shape[0], (batch_size,), device=device)
        real_batch = real_t[idx]

        # Update discriminator.
        z = torch.randn(batch_size, z_dim, device=device)
        fake_batch = G(z).detach()
        d_real = D(real_batch)
        d_fake = D(fake_batch)
        real_targets = torch.ones(batch_size, device=device) * 0.9
        fake_targets = torch.zeros(batch_size, device=device)
        d_loss = bce(d_real, real_targets) + bce(d_fake, fake_targets)
        opt_d.zero_grad()
        d_loss.backward()
        opt_d.step()

        # Update generator.
        z = torch.randn(batch_size, z_dim, device=device)
        fake_batch = G(z)
        d_fake_for_g = D(fake_batch)
        g_loss = bce(d_fake_for_g, torch.ones(batch_size, device=device))
        opt_g.zero_grad()
        g_loss.backward()
        opt_g.step()

        if epoch % 5 == 0 or epoch == 1:
            with torch.no_grad():
                history["d_loss"].append(float(d_loss.item()))
                history["g_loss"].append(float(g_loss.item()))
                history["d_real_score"].append(float(torch.sigmoid(d_real).mean().item()))
                history["d_fake_score"].append(float(torch.sigmoid(d_fake).mean().item()))

    return G, D, history


G2, D2, gan_history = train_gan_2d(real2_train_z)
plot_training_history(gan_history, "GAN training history", keys=["d_loss", "g_loss"])

In [ ]:
# Generate from the trained GAN.
def sample_gan_2d(G, n, z_dim=6, seed=SEED):
    set_seed(seed)
    G.eval()
    with torch.no_grad():
        z = torch.randn(n, z_dim, device=device)
        return G(z).cpu().numpy().astype(np.float32)


gan2_z = sample_gan_2d(G2, len(real2_test_z), seed=SEED + 4)
plot_2d(real2_test_z, gan2_z, "Real PCA-space profiles versus small GAN samples")

scores = pd.DataFrame(
    {
        "d_real_score": gan_history["d_real_score"],
        "d_fake_score": gan_history["d_fake_score"],
    }
)
print("Last discriminator scores. Values near 0.5 suggest the discriminator is less certain.")
display(scores.tail().round(3))

## 7. Mode collapse as missing coverage

Mode collapse occurs when a generator produces a narrow range of outputs that may look plausible but fails to cover the full data distribution. In business settings, this can erase rare segments, minority customer paths, edge cases, or low-frequency product combinations. A small set of attractive samples will not reveal the problem. Coverage metrics are needed.

In [ ]:
# ============================================================
# 7. Coverage and mode collapse diagnostics in PCA space
# ============================================================
real2_train_km = real2_train_z.astype(np.float64)
real2_test_km = real2_test_z.astype(np.float64)
kmeans = KMeans(n_clusters=NUM_SEGMENTS, random_state=SEED, n_init=20).fit(real2_train_km)
real_cluster = kmeans.predict(real2_test_km)
real_cluster_counts = np.bincount(real_cluster, minlength=NUM_SEGMENTS)
real_cluster_share = real_cluster_counts / real_cluster_counts.sum()

rng = np.random.default_rng(SEED)
mode0_pool = real2_train_km[kmeans.labels_ == 0]
collapsed = mode0_pool[rng.choice(len(mode0_pool), len(real2_test_z), replace=True)] + rng.normal(0, 0.04, size=real2_test_z.shape)

vae_for_coverage = real2_scaler.transform(pca_baseline.transform(vae_scaled)).astype(np.float32)
cvae_for_coverage = real2_scaler.transform(pca_baseline.transform(cvae_balanced_scaled[: len(real2_test_z)])).astype(np.float32)


def cluster_coverage_report(points, name):
    points = np.asarray(points, dtype=np.float64)
    assigned = kmeans.predict(points)
    counts = np.bincount(assigned, minlength=NUM_SEGMENTS)
    shares = counts / max(1, counts.sum())
    return {
        "generator": name,
        "missing_clusters": int((counts == 0).sum()),
        "coverage_entropy": entropy_from_shares(shares),
        "L1_share_gap_vs_real": float(np.abs(shares - real_cluster_share).sum()),
        "smallest_cluster_share": float(shares.min()),
    }

coverage_df = pd.DataFrame(
    [
        cluster_coverage_report(collapsed, "Deliberately collapsed generator"),
        cluster_coverage_report(gan2_z, "Small GAN"),
        cluster_coverage_report(vae_for_coverage, "VAE prior samples"),
        cluster_coverage_report(cvae_for_coverage, "cVAE requested samples"),
    ]
).round(3)
display(coverage_df)

In [ ]:
plot_2d(real2_test_z, collapsed, "Real PCA-space profiles versus deliberately collapsed samples")

## 8. Image-to-image translation as a problem setup

Many GAN applications are not just about creating new samples from noise. They learn transformations between domains. Paired translation uses matched examples, such as a sketch and its finished design. Unpaired translation uses separate domain collections and requires a constraint such as cycle consistency. The code below is only a visual analogy. It does not train pix2pix or CycleGAN, but it makes the data requirement clear.

In [ ]:
# ============================================================
# 8. Paired versus unpaired translation as a data setup
# ============================================================
rng = np.random.default_rng(SEED)
base = rng.normal(size=(80, 2))
rotation = np.array([[0.75, -0.45], [0.45, 0.75]])
paired_a = base
paired_b = base @ rotation.T + np.array([2.5, 0.5]) + rng.normal(scale=0.10, size=base.shape)

unpaired_a = rng.normal(loc=[-1.5, 0.0], scale=[0.65, 0.45], size=(80, 2))
unpaired_b = rng.normal(loc=[2.0, 0.6], scale=[0.55, 0.60], size=(80, 2))

plt.figure(figsize=(6.8, 5.2))
plt.scatter(paired_a[:, 0], paired_a[:, 1], s=18, alpha=0.55, label="Domain A")
plt.scatter(paired_b[:, 0], paired_b[:, 1], s=18, alpha=0.55, label="Domain B")
for i in range(0, len(paired_a), 4):
    plt.arrow(
        paired_a[i, 0],
        paired_a[i, 1],
        paired_b[i, 0] - paired_a[i, 0],
        paired_b[i, 1] - paired_a[i, 1],
        alpha=0.20,
        length_includes_head=True,
        head_width=0.04,
    )
plt.title("Paired translation: aligned examples are available")
plt.xlabel("Style dimension 1")
plt.ylabel("Style dimension 2")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

plt.figure(figsize=(6.8, 5.2))
plt.scatter(unpaired_a[:, 0], unpaired_a[:, 1], s=18, alpha=0.55, label="Domain A collection")
plt.scatter(unpaired_b[:, 0], unpaired_b[:, 1], s=18, alpha=0.55, label="Domain B collection")
plt.title("Unpaired translation: collections exist, but one-to-one matches do not")
plt.xlabel("Style dimension 1")
plt.ylabel("Style dimension 2")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 9. Evaluation scorecard: fidelity, coverage, alignment, and constraints

A generator should not be judged by a single number. Fidelity asks whether synthetic records resemble the real distribution. Coverage asks whether the generator represents the full support of the data, including less common regions. Alignment asks whether generation respects the intended condition or use case. Constraint checks ask whether synthetic records violate business rules, such as negative tenure or rates outside zero to one.

In [ ]:
# ============================================================
# 9. General evaluation metrics for tabular synthetic data
# ============================================================
BOUNDS = {
    "recency_days": (0, 180),
    "sessions_30d": (0, None),
    "orders_90d": (0, None),
    "log_monetary_90d": (0, None),
    "support_tickets_90d": (0, None),
    "discount_share": (0, 1),
    "email_open_rate": (0, 1),
    "mobile_share": (0, 1),
    "return_rate": (0, 1),
    "tenure_months": (0, 120),
}


def constraint_violation_rate(X_scaled):
    raw = raw_from_scaled(X_scaled)
    violation = np.zeros(len(raw), dtype=bool)
    for col, (lower, upper) in BOUNDS.items():
        if lower is not None:
            violation |= raw[col].values < lower
        if upper is not None:
            violation |= raw[col].values > upper
    return float(violation.mean())


def score_synthetic_tabular(name, X_synth, X_real=X_test_scaled, X_train_reference=X_train_scaled):
    X_synth = np.asarray(X_synth, dtype=np.float32)
    nn_synth = NearestNeighbors(n_neighbors=1).fit(X_synth)
    real_to_synth = nn_synth.kneighbors(X_real, return_distance=True)[0].ravel()

    nn_train = NearestNeighbors(n_neighbors=1).fit(X_train_reference)
    synth_to_train = nn_train.kneighbors(X_synth, return_distance=True)[0].ravel()

    return {
        "generator": name,
        "mean_gap": float(np.mean(np.abs(X_synth.mean(axis=0) - X_real.mean(axis=0)))),
        "std_gap": float(np.mean(np.abs(X_synth.std(axis=0) - X_real.std(axis=0)))),
        "corr_gap": float(np.mean(np.abs(safe_corr(X_synth) - safe_corr(X_real)))),
        "real_to_synth_dist_p95": float(np.quantile(real_to_synth, 0.95)),
        "constraint_violation_rate": constraint_violation_rate(X_synth),
        "nearest_train_dist_min": float(np.min(synth_to_train)),
        "nearest_train_dist_p01": float(np.quantile(synth_to_train, 0.01)),
    }

# cVAE samples with the same segment proportions as the training data.
rng = np.random.default_rng(SEED)
train_segment_share = np.bincount(y_train, minlength=NUM_SEGMENTS) / len(y_train)
cvae_eval_labels = rng.choice(np.arange(NUM_SEGMENTS), size=len(X_test_scaled), p=train_segment_share)
cvae_eval_scaled = sample_cvae(cvae, cvae_eval_labels, seed=SEED + 5)

scorecard = pd.DataFrame(
    [
        score_synthetic_tabular("Independent marginal baseline", independent_scaled),
        score_synthetic_tabular("VAE prior samples", vae_scaled),
        score_synthetic_tabular("cVAE prior samples", cvae_eval_scaled),
    ]
).round(4)
display(scorecard)

In [ ]:
# Save scorecard for later use.
scorecard_path = OUTPUT_DIR / "ch18_synthetic_data_scorecard.csv"
scorecard.to_csv(scorecard_path, index=False)
print(f"Saved: {scorecard_path}")

## 10. Utility validation with TSTR

A widely used utility check is TSTR: train on synthetic, test on real. The goal is not to prove that synthetic data is perfect. The goal is to ask whether synthetic data preserves enough decision-relevant structure to support a downstream task. Here the downstream task is segment classification. Because the cVAE was asked to generate profiles under requested segment labels, TSTR gives us a practical alignment check.

In [ ]:
# ============================================================
# 10. TSTR: train on synthetic, test on real
# ============================================================

def evaluate_segment_classifier(X_train_model, y_train_model, X_test_model=X_test_scaled, y_test_model=y_test, label="model"):
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train_model, y_train_model)
    pred = clf.predict(X_test_model)
    return {
        "training_source": label,
        "test_source": "real holdout",
        "accuracy": accuracy_score(y_test_model, pred),
        "macro_f1": f1_score(y_test_model, pred, average="macro"),
    }


def sample_independent_within_segments(X_reference, y_reference, n_per_class, seed=SEED):
    rng = np.random.default_rng(seed)
    samples = []
    labels = []
    for sid in range(NUM_SEGMENTS):
        X_seg = X_reference[y_reference == sid]
        block = []
        for j in range(X_reference.shape[1]):
            block.append(rng.choice(X_seg[:, j], size=n_per_class, replace=True))
        samples.append(np.column_stack(block))
        labels.extend([sid] * n_per_class)
    return np.vstack(samples).astype(np.float32), np.asarray(labels, dtype=int)

n_tstr_per_segment = 500 if FAST_MODE else 1000
tstr_labels = np.repeat(np.arange(NUM_SEGMENTS), n_tstr_per_segment)
X_cvae_tstr = sample_cvae(cvae, tstr_labels, seed=SEED + 6)
X_ind_tstr, y_ind_tstr = sample_independent_within_segments(X_train_scaled, y_train, n_tstr_per_segment, seed=SEED + 7)

utility_df = pd.DataFrame(
    [
        evaluate_segment_classifier(X_train_scaled, y_train, label="Real training data"),
        evaluate_segment_classifier(X_ind_tstr, y_ind_tstr, label="Independent marginals within segment"),
        evaluate_segment_classifier(X_cvae_tstr, tstr_labels, label="cVAE synthetic data"),
    ]
).round(4)
display(utility_df)

## 11. Privacy and memorization diagnostics

Synthetic data should be treated as derived data, not automatically anonymous data. A simple first diagnostic is a nearest-neighbor test: for each generated record, find the closest real training record. Very small distances do not prove a privacy violation by themselves, but they are red flags that require review. The copy generator below is intentionally unsafe and is included to calibrate the diagnostic.

In [ ]:
# ============================================================
# 11. Nearest-neighbor memorization diagnostics
# ============================================================

def nearest_train_report(X_synth, name, threshold=0.10):
    nn = NearestNeighbors(n_neighbors=1).fit(X_train_scaled)
    dist, idx = nn.kneighbors(X_synth, return_distance=True)
    dist = dist.ravel()
    return {
        "generator": name,
        "min_distance_to_train": float(np.min(dist)),
        "p01_distance_to_train": float(np.quantile(dist, 0.01)),
        "p05_distance_to_train": float(np.quantile(dist, 0.05)),
        "median_distance_to_train": float(np.median(dist)),
        f"share_below_{threshold}": float(np.mean(dist < threshold)),
    }

rng = np.random.default_rng(SEED)
copy_idx = rng.choice(len(X_train_scaled), size=len(X_test_scaled), replace=True)
copy_generator_scaled = X_train_scaled[copy_idx] + rng.normal(scale=0.005, size=X_test_scaled.shape).astype(np.float32)

privacy_df = pd.DataFrame(
    [
        nearest_train_report(copy_generator_scaled, "Unsafe copy generator"),
        nearest_train_report(independent_scaled, "Independent marginal baseline"),
        nearest_train_report(vae_scaled, "VAE prior samples"),
        nearest_train_report(cvae_eval_scaled, "cVAE prior samples"),
    ]
).round(4)
display(privacy_df)

In [ ]:
# Inspect one closest cVAE case and its nearest real training neighbor.
nn_train = NearestNeighbors(n_neighbors=1).fit(X_train_scaled)
dist, idx = nn_train.kneighbors(cvae_eval_scaled, return_distance=True)
closest_synth_idx = int(np.argmin(dist.ravel()))
closest_train_idx = int(idx[closest_synth_idx, 0])
comparison = pd.DataFrame(
    {
        "cVAE_synthetic": raw_from_scaled(cvae_eval_scaled[[closest_synth_idx]]).iloc[0][feature_cols],
        "nearest_real_train": df_train.iloc[closest_train_idx][feature_cols],
    }
).round(3)
print("Closest cVAE synthetic record versus nearest real training record:")
display(comparison)

## 12. Bridge to diffusion models: one-step generation versus iterative denoising

VAEs and GANs usually produce a sample in one pass after training. Diffusion models take a different path. They learn to reverse a gradual noising process, turning noise into data through many smaller denoising steps. The figure below is a toy illustration, not a trained diffusion model. It previews why the next chapter treats diffusion as a different answer to the question of how a model learns realism.

In [ ]:
# ============================================================
# 12. Toy illustration of iterative denoising
# ============================================================
rng = np.random.default_rng(SEED)
x0 = real2_test_z[rng.integers(0, len(real2_test_z))]
noise = rng.normal(loc=0, scale=2.0, size=2)
steps = np.linspace(0, 1, 9)
forward_path = np.array([(1 - a) * x0 + a * noise for a in steps])
reverse_path = np.array([(1 - a) * noise + a * x0 for a in steps])

plt.figure(figsize=(7, 5.5))
plt.scatter(real2_test_z[:, 0], real2_test_z[:, 1], s=10, alpha=0.12, label="Real profile cloud")
plt.plot(forward_path[:, 0], forward_path[:, 1], marker="o", label="Forward noising path")
plt.plot(reverse_path[:, 0], reverse_path[:, 1], marker="x", label="Reverse denoising path")
plt.scatter([x0[0]], [x0[1]], s=80, label="Data point")
plt.scatter([noise[0]], [noise[1]], s=80, label="Noise point")
plt.title("Toy bridge to diffusion: data to noise, then noise to data")
plt.xlabel("PCA-space dimension 1")
plt.ylabel("PCA-space dimension 2")
plt.legend(fontsize=8)
plt.grid(alpha=0.2)
plt.show()

## 13. Governance artifacts

A generative model should leave evidence behind. The evidence does not need to be long, but it should be concrete enough for another analyst, manager, or reviewer to understand the intended use, known limits, evaluation results, privacy checks, and approval boundaries.

In [ ]:
# ============================================================
# 13. Lightweight governance card and saved artifacts
# ============================================================
model_card = {
    "chapter": "Chapter 18 Classic Generative Models",
    "model_family": "Conditional VAE teaching example",
    "intended_use": "Generate synthetic customer profiles for internal simulation and classroom demonstration.",
    "not_for": [
        "Treating synthetic records as real customers",
        "External data sharing without additional privacy review",
        "Causal claims about interventions",
        "Automated decisions affecting real customers",
    ],
    "training_data": "Synthetic customer profiles generated inside this notebook",
    "conditions_supported": SEGMENTS,
    "evaluation_artifacts": [
        str(scorecard_path),
        "TSTR utility table displayed in notebook",
        "Nearest-neighbor memorization table displayed in notebook",
    ],
    "main_risks": [
        "Coverage gaps across segments",
        "Constraint violations in generated profiles",
        "Overtrust in plausible-looking synthetic data",
        "Potential memorization if trained on sensitive real data",
    ],
    "required_review_before_real_use": [
        "Data provenance review",
        "Privacy and confidentiality review",
        "Coverage report tied to business segments",
        "Human review protocol for purpose alignment",
        "Versioning and re-test schedule",
    ],
}

model_card_path = OUTPUT_DIR / "ch18_generative_model_card.json"
with open(model_card_path, "w") as f:
    json.dump(model_card, f, indent=2)

# Save one synthetic sample file for inspection.
sample_export = cvae_balanced_raw.sample(100, random_state=SEED).reset_index(drop=True)
sample_path = OUTPUT_DIR / "ch18_cvae_synthetic_customer_sample.csv"
sample_export.to_csv(sample_path, index=False)

print(f"Saved: {model_card_path}")
print(f"Saved: {sample_path}")
print(json.dumps(model_card, indent=2))

## Decision guide

Use the VAE family when the latent representation is itself valuable, when stable training matters, or when smooth controlled variation is more important than maximum visual sharpness. Use the GAN family when high-fidelity synthesis or domain transformation is central and the team has enough time to diagnose instability, mode collapse, and memorization risk. In both cases, avoid judging a generator by a few attractive examples. The practical question is whether the generated distribution improves a specific business workflow under realistic constraints.

## Exercises

1. Change `LATENT_DIM` from 4 to 2 or 8, rerun the VAE and cVAE sections, and compare the scorecard. Does a larger latent space improve fidelity, coverage, or TSTR utility?

2. Change the VAE `beta_max` value in `train_vae`. What happens when the KL regularization pressure is weaker or stronger?

3. Generate cVAE samples for only one segment. Does the generated profile match that segment better on engagement, monetary value, or service friction?

4. Increase `EPOCHS_GAN` and rerun the GAN section. Does the GAN improve coverage, or does it become unstable?

5. Create a deliberately biased generator by oversampling only the `Loyal Premium` segment. How does the coverage table change?

6. Tighten the nearest-neighbor threshold in the privacy diagnostic from `0.10` to `0.05`. Which generators remain safe under the stricter threshold?

7. Add one business constraint to `BOUNDS`, such as a maximum plausible order count or a maximum support ticket count. Which generator violates the new rule most often?

8. Replace segment classification in the TSTR section with churn prediction. What would be needed to generate both synthetic features and synthetic labels responsibly?

## Wrap-up

This notebook showed how classic generative models become business analytics tools only after validation. VAEs provide a smooth sampleable latent space by balancing reconstruction and regularization. Conditional VAEs add control. GANs learn from an adversarial discriminator and can produce realistic samples, but require careful monitoring for instability and mode collapse. The main lesson is stable across model families: synthetic data is useful only when it is faithful enough, diverse enough, aligned with purpose, and governed as derived data.